In [13]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# LangSmith tracing (v1 SDK uses LANGSMITH_* names)
_LS_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
_LS_PROJECT = os.getenv("LANGSMITH_PROJECT") or os.getenv("LANGCHAIN_PROJECT", "Simple-GenAI-App")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = _LS_KEY
os.environ["LANGSMITH_PROJECT"] = _LS_PROJECT

assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY missing in .env"
assert os.environ["LANGSMITH_API_KEY"], "LANGSMITH_API_KEY (or LANGCHAIN_API_KEY) missing in .env"
print("Tracing project:", os.environ["LANGSMITH_PROJECT"])

Tracing project: GENERATIVE_AIAPPWITHOPENAI


In [14]:
###Data Ingestion-- From the Website We Need to Scrap the Data
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [15]:
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
loader

In [16]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nArtificial intelligence - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nGoals\n\n\n\n\nToggle Goals subsectio

In [17]:
###Load Data-->Docs-->Divide out Text into chunks -->text-->Vector-->Vector Embeddings-->Vector Store-->Retrieval-->LLM

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [9]:
documents

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}, page_content='Artificial intelligence - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nGoals\n\n\n\n\nToggle Goals subsection\n\n\n\

In [18]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [19]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [20]:
vectorstoredb

In [21]:
##Query from Vector Store DB
query = "What is Artificial Intelligence?"
result = vectorstoredb.similarity_search(query)
result[0].page_content

'Retrieved from "https://en.wikipedia.org/w/index.php?title=Artificial_intelligence&oldid=1354067166"\nCategories: Artificial intelligenceComputational fields of studyComputational neuroscienceCyberneticsData scienceFormal sciencesIntelligence by typeHidden categories: Webarchive template wayback linksCS1 German-language sources (de)CS1 Russian-language sources (ru)CS1 Japanese-language sources (ja)CS1: long volume valueArticles with short descriptionShort description is different from WikidataUse dmy dates from October 2025Wikipedia indefinitely semi-protected pagesArticles with excerptsPages displaying short descriptions of redirect targets via Module:Annotated linkArticles with Internet Encyclopedia of Philosophy linksPages using Sister project links with hidden wikidata'

In [22]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.o")

In [23]:
##Retrival Chain,Document Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on provided context:
<context>
{context}
</context>
"""
)

documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
documents_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on provided context:\n<context>\n{context}\n</context>\n'), additional_kwargs={})])
| ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x1680641c0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1633e3730>, root_client=<openai.OpenAI object at 0x15e7fee60>, root_async_client=<openai.AsyncOpenAI object at 0x168064130>, model_name='gpt-4.o', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True, stream_chunk_

In [24]:
from langchain_core.documents import Document
documents_chain.invoke({
    "input":"What is Artificial Intelligence?",
    "context": [Document(page_content="Artificial Intelligence is a branch of computer science that aims to create software or machines that exhibit human-like intelligence.")]
})

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-4.o` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [25]:
vectorstoredb

In [26]:
vectorstoredb.as_retriever()

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1633e3580>, search_kwargs={})

In [29]:
retriever = vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever,documents_chain)

ModuleNotFoundError: No module named 'langchain.chains'

In [30]:
retrieval_chain

NameError: name 'retrieval_chain' is not defined

In [ ]:
###Get the answer from the Retrieval Chain LLM
response = retrieval_chain.invoke("What is Artificial Intelligence?")
